# Multi-Player NBA Points Predictor

Generalised XGBoost model trained on 31 star players (2018-19 → 2024-25).

**Why multi-player?**
- Single-player models (~400 games) are too small for XGBoost — it memorises rather than generalises
- 31 players gives ~15,000+ games, forcing the model to learn patterns that hold across different players, teams and contexts
- Structural breaks (trades, team changes) appear across many players, so the model can learn to handle them

**Features:**
- `season_avg_so_far` — expanding mean within the season (captures player identity + current form baseline)
- `form_vs_season` — roll5_pts minus season_avg_so_far (hot/cold relative to own baseline)
- `roll5_pts`, `roll10_pts` — short and medium-term form
- `roll5_ast`, `roll5_tov` — usage signals
- `HOME`, `b2b`, `days_rest` — context
- `opp_def_rating`, `opp_off_rating` — opponent quality both ends
- `games_on_new_team` — trade adjustment indicator

In [ ]:
import glob, json, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from nba_api.stats.static import teams as nba_teams
from nba_api.stats.endpoints import playergamelogs

# ── 31-player roster ──────────────────────────────────────────────────────────
PLAYERS = {
    2544:    'LeBron James',
    201939:  'Stephen Curry',
    201142:  'Kevin Durant',
    201935:  'James Harden',
    202695:  'Kawhi Leonard',
    203507:  'Giannis Antetokounmpo',
    203999:  'Nikola Jokic',
    203954:  'Joel Embiid',
    1628369: 'Jayson Tatum',
    202710:  'Jimmy Butler',
    202331:  'Paul George',
    203081:  'Damian Lillard',
    101108:  'Chris Paul',
    201566:  'Russell Westbrook',
    202681:  'Kyrie Irving',
    1626164: 'Devin Booker',
    203076:  'Anthony Davis',
    1626157: 'Karl-Anthony Towns',
    203497:  'Rudy Gobert',
    1628378: 'Donovan Mitchell',
    1629027: 'Trae Young',
    1629630: 'Ja Morant',
    1629627: 'Zion Williamson',
    1628983: 'Shai Gilgeous-Alexander',
    1630162: 'Anthony Edwards',
    1641705: 'Victor Wembanyama',
    1627759: 'Jaylen Brown',
    203078:  'Bradley Beal',
    201942:  'DeMar DeRozan',
    1627783: 'Pascal Siakam',
    1629029: 'Luka Doncic',
}

SEASONS = {
    '2018-19': '2018_19', '2019-20': '2019_20', '2020-21': '2020_21',
    '2021-22': '2021_22', '2022-23': '2022_23', '2023-24': '2023_24',
    '2024-25': '2024_25',
}

print(f"Roster: {len(PLAYERS)} players | Seasons: {len(SEASONS)}")

## Step 1 — Fetch Game Logs

Fetch every game log for every player across all seasons from `nba_api`.  
Takes ~10 minutes due to rate-limit sleeps (~217 player-season combinations).  
**After the first run, skip this cell and use the Load cell below.**

In [ ]:
RAW_PATH = '../data/processed/multi_player_logs.json'

all_logs = {}
total = len(PLAYERS) * len(SEASONS)
done  = 0

for pid, name in PLAYERS.items():
    all_logs[str(pid)] = {}
    for api_season, bbref_key in SEASONS.items():
        try:
            logs = playergamelogs.PlayerGameLogs(
                player_id_nullable=pid,
                season_nullable=api_season,
            ).get_data_frames()[0]

            if not logs.empty:
                keep = [c for c in ['GAME_DATE','TEAM_ABBREVIATION','MATCHUP',
                        'WL','MIN','PTS','REB','AST','TOV','STL','BLK',
                        'FGM','FGA','FG_PCT','FG3M','FG3A','FG3_PCT',
                        'FTM','FTA','FT_PCT','PLUS_MINUS'] if c in logs.columns]
                mini = logs[keep].copy()
                mini['GAME_DATE'] = pd.to_datetime(mini['GAME_DATE']).dt.strftime('%Y-%m-%d')
                all_logs[str(pid)][bbref_key] = mini.to_dict('records')
        except Exception as e:
            print(f"  ERROR {name} {api_season}: {e}")

        done += 1
        if done % 20 == 0:
            print(f"  {done}/{total} fetched...")
        time.sleep(0.65)

with open(RAW_PATH, 'w') as f:
    json.dump(all_logs, f)

total_games = sum(len(r) for pd_ in all_logs.values() for r in pd_.values())
print(f"Saved {RAW_PATH} — {total_games} game records")

In [ ]:
# Load cached game logs (run this instead of the fetch cell on future sessions)
RAW_PATH = '../data/processed/multi_player_logs.json'

with open(RAW_PATH) as f:
    raw = json.load(f)

player_dfs = {}
for pid_str, season_dict in raw.items():
    pid = int(pid_str)
    player_dfs[pid] = {}
    for season_key, records in season_dict.items():
        if records:
            df = pd.DataFrame(records)
            df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
            player_dfs[pid][season_key] = df

total_games = sum(len(df) for pd_ in player_dfs.values() for df in pd_.values())
print(f"Loaded {len(player_dfs)} players — {total_games} total game records")

## Step 2 — Opponent Ratings (DEF + OFF)

Fetch both defensive and offensive ratings per team per season from `nba_api`.  
Save separately so the fetch only runs once.

In [ ]:
# Fetch and save opponent OFF + DEF ratings
# Skip this cell on future runs — use the load cell below
from nba_api.stats.endpoints import leaguedashteamstats

SEASON_API = {v: k for k, v in SEASONS.items()}
RATINGS_PATH = '../data/processed/team_ratings.json'

team_ratings = {}   # {bbref_key: {team_id: {'DEF': x, 'OFF': x}}}

for bbref_key, api_season in SEASON_API.items():
    print(f"Fetching {api_season}...")
    df_stats = leaguedashteamstats.LeagueDashTeamStats(
        season=api_season,
        measure_type_detailed_defense='Advanced',
    ).get_data_frames()[0]

    team_ratings[bbref_key] = {
        str(int(row['TEAM_ID'])): {
            'DEF': row['DEF_RATING'],
            'OFF': row['OFF_RATING'],
        }
        for _, row in df_stats.iterrows()
    }
    time.sleep(0.6)

with open(RATINGS_PATH, 'w') as f:
    json.dump(team_ratings, f)
print(f"Saved {RATINGS_PATH}")

In [ ]:
# Load cached team ratings
RATINGS_PATH = '../data/processed/team_ratings.json'

with open(RATINGS_PATH) as f:
    raw_ratings = json.load(f)

# {bbref_key: {team_id (int): {'DEF': float, 'OFF': float}}}
team_ratings = {
    s: {int(k): v for k, v in d.items()}
    for s, d in raw_ratings.items()
}

_static        = nba_teams.get_teams()
NBA_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in _static}

print("Team ratings loaded.")
print(f"Seasons: {list(team_ratings.keys())}")

## Step 3 — Feature Engineering

For each player-game:
- Rolling lag features shifted by 1 (no leakage)
- **Trade detection**: `games_on_new_team` resets when `TEAM_ABBREVIATION` changes mid-season — captures the adjustment period after a trade
- Opponent DEF and OFF ratings via team_id lookup
- Season baseline (`season_avg_so_far`) and relative form (`form_vs_season`)

In [ ]:
def extract_opp_abbr(matchup_str):
    """Pull opponent abbreviation from 'GSW vs. LAL' or 'GSW @ LAL'."""
    parts = re.split(r'\s+(?:vs\.|@)\s+', str(matchup_str))
    return parts[1].strip() if len(parts) == 2 else None

def build_features(pid, season_key, df_in):
    df = df_in.copy().sort_values('GAME_DATE').reset_index(drop=True)

    # Trade detection
    df['team_changed'] = (df['TEAM_ABBREVIATION'] != df['TEAM_ABBREVIATION'].shift(1)).astype(int)
    df.loc[0, 'team_changed'] = 0
    df['trade_block']       = df['team_changed'].cumsum()
    df['games_on_new_team'] = df.groupby('trade_block').cumcount()

    # Context
    df['HOME']      = (~df['MATCHUP'].str.contains('@')).astype(int)
    df['days_rest'] = df['GAME_DATE'].diff().dt.days.clip(upper=14).fillna(2)
    df['b2b']       = (df['days_rest'] == 1).astype(int)

    # Numeric
    for col in ['PTS', 'AST', 'TOV', 'REB', 'MIN']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Rolling lag features
    df['roll5_pts']  = df['PTS'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll10_pts'] = df['PTS'].shift(1).rolling(10, min_periods=5).mean()
    df['roll5_ast']  = df['AST'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll5_tov']  = df['TOV'].shift(1).rolling(5,  min_periods=3).mean()

    # Season baseline
    df['season_avg_so_far'] = df['PTS'].shift(1).expanding().mean()
    df['form_vs_season']    = df['roll5_pts'] - df['season_avg_so_far']

    # Opponent ratings
    def opp_rating(row, key):
        abbr = extract_opp_abbr(row['MATCHUP'])
        if not abbr:
            return np.nan
        tid = NBA_ABBR_TO_ID.get(abbr)
        if not tid:
            return np.nan
        return team_ratings.get(season_key, {}).get(tid, {}).get(key, np.nan)

    df['opp_def_rating'] = df.apply(lambda r: opp_rating(r, 'DEF'), axis=1)
    df['opp_off_rating'] = df.apply(lambda r: opp_rating(r, 'OFF'), axis=1)

    df['player_id']   = pid
    df['player_name'] = PLAYERS[pid]
    df['season_key']  = season_key
    return df

# Build full dataset
all_dfs = []
for pid, season_dict in player_dfs.items():
    for season_key, df_season in season_dict.items():
        try:
            all_dfs.append(build_features(pid, season_key, df_season))
        except Exception as e:
            print(f"  ERROR {PLAYERS.get(pid)} {season_key}: {e}")

df_all = pd.concat(all_dfs, ignore_index=True).sort_values('GAME_DATE').reset_index(drop=True)
print(f"Full dataset: {len(df_all)} rows | {df_all['player_name'].nunique()} players")
print(f"Date range: {df_all['GAME_DATE'].min().date()} → {df_all['GAME_DATE'].max().date()}")

In [ ]:
FEATURES = [
    'season_avg_so_far', 'form_vs_season',
    'roll5_pts', 'roll10_pts',
    'roll5_ast', 'roll5_tov',
    'HOME', 'b2b', 'days_rest',
    'opp_def_rating', 'opp_off_rating',
    'games_on_new_team',
]

df_model = df_all.dropna(subset=FEATURES + ['PTS']).copy().reset_index(drop=True)

print(f"Model-ready: {len(df_model)} rows | {df_model['player_name'].nunique()} players")
print(f"\nGames per player:")
print(df_model.groupby('player_name')['PTS'].count().sort_values(ascending=False).to_string())
print(f"\nFeature summary:")
df_model[FEATURES + ['PTS']].describe().round(2)

## Step 4 — Train / Test Split

Train on 2018-19 through end of 2022-23.  
Test on 2023-24 and 2024-25 — two full seasons including many high-profile trades across the roster.  
This is a strict chronological split — no future data leaks into training.

In [ ]:
SPLIT_DATE = pd.Timestamp('2023-10-01')

train_mask = df_model['GAME_DATE'] < SPLIT_DATE
test_mask  = ~train_mask

X_train = df_model.loc[train_mask, FEATURES].values
y_train = df_model.loc[train_mask, 'PTS'].values
X_test  = df_model.loc[test_mask,  FEATURES].values
y_test  = df_model.loc[test_mask,  'PTS'].values

print(f"Train: {len(X_train):,} games  up to {df_model.loc[train_mask,'GAME_DATE'].max().date()}")
print(f"Test : {len(X_test):,}  games  from {df_model.loc[test_mask, 'GAME_DATE'].min().date()}")
print(f"\nTrain mean PTS: {y_train.mean():.1f}  |  Test mean PTS: {y_test.mean():.1f}")

## Step 5 — XGBoost Model

Edit `PARAMS` and re-run to experiment.

| Parameter | What it controls | Range |
|---|---|---|
| `n_estimators` | Number of trees | 100–1000 |
| `max_depth` | Tree depth — deeper = more complex | 2–8 |
| `learning_rate` | Contribution per tree | 0.01–0.3 |
| `subsample` | Row sampling per tree | 0.5–1.0 |
| `colsample_bytree` | Feature sampling per tree | 0.5–1.0 |
| `min_child_weight` | Min samples per leaf — higher = more conservative | 1–20 |
| `reg_alpha` | L1 regularisation | 0–2 |
| `reg_lambda` | L2 regularisation | 0–5 |

**Tip:** with ~15k rows you can afford deeper trees and more estimators than the single-player model.

In [ ]:
# ── Edit and re-run ────────────────────────────────────────────────────────────
PARAMS = dict(
    n_estimators     = 500,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 10,
    reg_alpha        = 0.5,
    reg_lambda       = 2.0,
    objective        = 'reg:squarederror',
    random_state     = 42,
    n_jobs           = -1,
)
# ───────────────────────────────────────────────────────────────────────────────

model = xgb.XGBRegressor(**PARAMS)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae   = mean_absolute_error(y_test, y_pred)
test_r2    = r2_score(y_test, y_pred)

print("=== Multi-Player Model Metrics ===")
print(f"  Train RMSE : {train_rmse:.2f} pts")
print(f"  Test  R2   : {test_r2:.3f}")
print(f"  Test  RMSE : {test_rmse:.2f} pts")
print(f"  Test  MAE  : {test_mae:.2f} pts")
print(f"  Gap        : {abs(train_rmse-test_rmse):.2f} pts",
      '(overfitting)' if train_rmse < test_rmse - 1 else '(stable)')

# Per-player RMSE
test_df = df_model[test_mask].copy()
test_df['pred']     = y_pred
test_df['residual'] = y_test - y_pred

per_player = (
    test_df.groupby('player_name')
    .apply(lambda g: pd.Series({
        'games': len(g),
        'avg_pts': g['PTS'].mean().round(1),
        'rmse': round(np.sqrt(mean_squared_error(g['PTS'], g['pred'])), 2),
        'mae':  round(mean_absolute_error(g['PTS'], g['pred']), 2),
    }))
    .sort_values('rmse', ascending=False)
)
print(f"\n=== Per-Player RMSE ===")
print(per_player.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.15, edgecolors='none', color='steelblue', s=10)
lims = [0, max(y_test.max(), y_pred.max()) + 2]
ax.plot(lims, lims, 'r--', lw=1.5, label='perfect')
ax.set_xlabel('Actual PTS'); ax.set_ylabel('Predicted PTS')
ax.set_title(f'Actual vs Predicted  (RMSE={test_rmse:.2f} pts)')
ax.legend()

# Per-player RMSE
ax = axes[1]
pp = per_player.sort_values('rmse')
ax.barh(pp.index, pp['rmse'], color='steelblue', alpha=0.7)
ax.axvline(test_rmse, color='red', linestyle='--', lw=1.5,
           label=f'Overall ({test_rmse:.2f})')
ax.set_title('Per-Player RMSE (test set)')
ax.set_xlabel('RMSE (pts)')
ax.legend()

# Feature importance
ax = axes[2]
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance (gain)')
ax.set_xlabel('Importance score')

plt.tight_layout()
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
# Sample 3000 rows for speed
idx = np.random.choice(len(X_train), min(3000, len(X_train)), replace=False)
shap_values = explainer.shap_values(X_train[idx])

shap.summary_plot(shap_values, X_train[idx],
                  feature_names=FEATURES, plot_type='dot', show=True)

## Step 6 — Quantile Regression (Prediction Intervals)

Train three models for the 10th, 50th and 90th percentiles to produce an 80% prediction interval.  
Coverage = % of actual games that fall within the interval (target: 80%).

In [ ]:
q_params = {**PARAMS, 'objective': 'reg:quantileerror', 'n_jobs': -1}

model_q10 = xgb.XGBRegressor(**{**q_params, 'quantile_alpha': 0.10})
model_q50 = xgb.XGBRegressor(**{**q_params, 'quantile_alpha': 0.50})
model_q90 = xgb.XGBRegressor(**{**q_params, 'quantile_alpha': 0.90})

for m in (model_q10, model_q50, model_q90):
    m.fit(X_train, y_train)

pred_q10 = model_q10.predict(X_test)
pred_q50 = model_q50.predict(X_test)
pred_q90 = model_q90.predict(X_test)

within    = (y_test >= pred_q10) & (y_test <= pred_q90)
coverage  = within.mean() * 100
avg_width = (pred_q90 - pred_q10).mean()

print("=== Quantile Regression (80% prediction interval) ===")
print(f"  Median RMSE       : {np.sqrt(mean_squared_error(y_test, pred_q50)):.2f} pts")
print(f"  Coverage          : {coverage:.1f}%  (target: 80%)")
print(f"  Avg interval width: {avg_width:.1f} pts")

test_df['q10']    = pred_q10
test_df['q90']    = pred_q90
test_df['within'] = within

per_cov = (
    test_df.groupby('player_name')['within']
    .mean().mul(100).round(1)
    .sort_values()
    .rename('coverage_%')
)
print(f"\n=== Per-Player Coverage ===")
print(per_cov.to_string())

## Step 7 — Walk-Forward Validation

Expanding-window validation: train on all data up to month M, test on month M+1.  
Gives the most honest out-of-sample RMSE across the full history.

In [ ]:
MIN_TRAIN = 3000

dates_wf = df_model['GAME_DATE']
X_wf     = df_model[FEATURES].values
y_wf     = df_model['PTS'].values

months   = sorted(dates_wf.dt.to_period('M').unique())
wf_rows  = []

for test_month in months:
    tr_mask = dates_wf.dt.to_period('M') < test_month
    te_mask = dates_wf.dt.to_period('M') == test_month
    if tr_mask.sum() < MIN_TRAIN or te_mask.sum() == 0:
        continue
    m = xgb.XGBRegressor(**PARAMS)
    m.fit(X_wf[tr_mask], y_wf[tr_mask])
    preds = m.predict(X_wf[te_mask])
    for date, actual, pred, pname in zip(
        dates_wf[te_mask], y_wf[te_mask], preds,
        df_model.loc[te_mask, 'player_name']
    ):
        wf_rows.append({'date': date, 'actual': actual,
                        'pred': pred, 'player': pname,
                        'month': str(test_month)})

wf_df   = pd.DataFrame(wf_rows)
wf_rmse = np.sqrt(mean_squared_error(wf_df['actual'], wf_df['pred']))
wf_r2   = r2_score(wf_df['actual'], wf_df['pred'])

print(f"Walk-forward: {len(wf_df):,} predictions across {wf_df['month'].nunique()} months")
print(f"  RMSE : {wf_rmse:.2f} pts")
print(f"  MAE  : {mean_absolute_error(wf_df['actual'], wf_df['pred']):.2f} pts")
print(f"  R2   : {wf_r2:.3f}")

monthly = (
    wf_df.groupby('month')
    .apply(lambda g: np.sqrt(mean_squared_error(g['actual'], g['pred'])))
    .reset_index(name='rmse')
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(monthly)), monthly['rmse'], color='steelblue', alpha=0.7)
ax.axhline(wf_rmse, color='red', linestyle='--', lw=1.5,
           label=f'Overall RMSE ({wf_rmse:.2f})')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['month'], rotation=45, ha='right', fontsize=7)
ax.set_title('Walk-Forward RMSE by Month (31-player model)')
ax.set_ylabel('RMSE (pts)')
ax.legend()
plt.tight_layout()
plt.show()

## Step 8 — Enhanced Features: Minutes Played + Rolling Opponent Ratings

Two new features added on top of the existing model:

- **`roll5_min`** — 5-game rolling average of minutes played (lag-1). Captures load management:  
  a player restricted to 20 minutes is systematically over-predicted without this.
- **`opp_def_rating` / `opp_off_rating` (rolling)** — monthly snapshots of opponent team  
  ratings replace the static season average. Early-season predictions previously relied on  
  last season's numbers; these now update each month as the new season plays out.

Fetch rolling ratings once, then use the load cell on future runs.  
All downstream cells (`df_model`, `X_train`, etc.) are overridden so existing cells re-use new data.

In [ ]:
# Fetch monthly-snapshot team ratings — skip on future runs, use load cell below
# 7 seasons × 6 monthly snapshots = 42 API calls (~30 seconds)
from nba_api.stats.endpoints import leaguedashteamstats

ROLLING_RATINGS_PATH = '../data/processed/team_ratings_rolling.json'

SEASON_START_YEAR = {
    '2018_19': 2018, '2019_20': 2019, '2020_21': 2020,
    '2021_22': 2021, '2022_23': 2022, '2023_24': 2023, '2024_25': 2024,
}
SEASON_API_MAP = {v: k for k, v in SEASONS.items()}   # bbref_key -> api_season

rolling_ratings_raw = {}   # {bbref_key: {snapshot_date: {team_id_str: {DEF, OFF}}}}

for bbref_key, api_season in SEASON_API_MAP.items():
    start_yr = SEASON_START_YEAR[bbref_key]
    rolling_ratings_raw[bbref_key] = {}
    # Nov of start year through Apr of end year
    snapshot_dates = [
        f"{start_yr}-11-01", f"{start_yr}-12-01",
        f"{start_yr+1}-01-01", f"{start_yr+1}-02-01",
        f"{start_yr+1}-03-01", f"{start_yr+1}-04-01",
    ]
    for date_to in snapshot_dates:
        try:
            df_snap = leaguedashteamstats.LeagueDashTeamStats(
                season=api_season,
                measure_type_detailed_defense='Advanced',
                date_to_nullable=date_to,
            ).get_data_frames()[0]
            if not df_snap.empty:
                rolling_ratings_raw[bbref_key][date_to] = {
                    str(int(row['TEAM_ID'])): {
                        'DEF': row['DEF_RATING'],
                        'OFF': row['OFF_RATING'],
                    }
                    for _, row in df_snap.iterrows()
                }
                print(f"  {api_season}  {date_to}  — {len(df_snap)} teams")
        except Exception as e:
            print(f"  ERROR {api_season} {date_to}: {e}")
        time.sleep(0.65)

with open(ROLLING_RATINGS_PATH, 'w') as f:
    json.dump(rolling_ratings_raw, f)
print(f"Saved {ROLLING_RATINGS_PATH}")


In [ ]:
# Load cached rolling team ratings
ROLLING_RATINGS_PATH = '../data/processed/team_ratings_rolling.json'

with open(ROLLING_RATINGS_PATH) as f:
    _raw_roll = json.load(f)

# {bbref_key: {snapshot_date_str: {team_id (int): {DEF, OFF}}}}
rolling_ratings = {
    season: {
        snap_date: {int(k): v for k, v in snap.items()}
        for snap_date, snap in snaps.items()
    }
    for season, snaps in _raw_roll.items()
}

print("Rolling ratings loaded.")
for s, snaps in rolling_ratings.items():
    print(f"  {s}: {len(snaps)} snapshots  ({', '.join(sorted(snaps)[:2])} ...)")


In [ ]:
import math

def parse_min(val):
    """Handle both '35:23' and 35.38 formats from nba_api."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return np.nan
    s = str(val).strip()
    if ':' in s:
        parts = s.split(':')
        try:
            return float(parts[0]) + float(parts[1]) / 60
        except ValueError:
            return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

def get_rolling_opp_rating(game_date, season_key, opp_abbr, key):
    """
    Return opponent rating from the most recent monthly snapshot
    that predates game_date.  Falls back to season-level rating if no
    snapshot exists yet (e.g. October games at season start).
    """
    tid = NBA_ABBR_TO_ID.get(opp_abbr)
    if not tid:
        return np.nan
    season_snaps = rolling_ratings.get(season_key, {})
    eligible = [d for d in season_snaps if pd.Timestamp(d) <= game_date]
    if eligible:
        latest = max(eligible)
        return season_snaps[latest].get(tid, {}).get(key, np.nan)
    # Fall back to static season rating for early-season games
    return team_ratings.get(season_key, {}).get(tid, {}).get(key, np.nan)

def build_features_v2(pid, season_key, df_in):
    df = df_in.copy().sort_values('GAME_DATE').reset_index(drop=True)

    # Trade detection
    df['team_changed'] = (df['TEAM_ABBREVIATION'] != df['TEAM_ABBREVIATION'].shift(1)).astype(int)
    df.loc[0, 'team_changed'] = 0
    df['trade_block']       = df['team_changed'].cumsum()
    df['games_on_new_team'] = df.groupby('trade_block').cumcount()

    # Context
    df['HOME']      = (~df['MATCHUP'].str.contains('@')).astype(int)
    df['days_rest'] = df['GAME_DATE'].diff().dt.days.clip(upper=14).fillna(2)
    df['b2b']       = (df['days_rest'] == 1).astype(int)

    # Parse minutes
    df['MIN_num'] = df['MIN'].apply(parse_min)
    for col in ['PTS', 'AST', 'TOV']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Rolling lag features (shift=1 prevents leakage)
    df['roll5_pts']  = df['PTS'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll10_pts'] = df['PTS'].shift(1).rolling(10, min_periods=5).mean()
    df['roll5_ast']  = df['AST'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll5_tov']  = df['TOV'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll5_min']  = df['MIN_num'].shift(1).rolling(5, min_periods=3).mean()  # NEW

    # Season baseline
    df['season_avg_so_far'] = df['PTS'].shift(1).expanding().mean()
    df['form_vs_season']    = df['roll5_pts'] - df['season_avg_so_far']

    # Opponent rolling ratings (monthly snapshot — no leakage)
    def opp_roll_rating(row, key):
        abbr = extract_opp_abbr(row['MATCHUP'])
        if not abbr:
            return np.nan
        return get_rolling_opp_rating(row['GAME_DATE'], season_key, abbr, key)

    df['opp_def_rating'] = df.apply(lambda r: opp_roll_rating(r, 'DEF'), axis=1)
    df['opp_off_rating'] = df.apply(lambda r: opp_roll_rating(r, 'OFF'), axis=1)

    df['player_id']   = pid
    df['player_name'] = PLAYERS[pid]
    df['season_key']  = season_key
    return df

print("build_features_v2 defined.")
print("  New features: roll5_min (minutes workload), rolling opp_def_rating, rolling opp_off_rating")


In [ ]:
# Rebuild full dataset with v2 features
# This overrides df_all, df_model, and FEATURES so existing downstream cells
# (train/test split, model, SHAP, quantile, walk-forward) pick up the new features.

all_dfs_v2 = []
for pid, season_dict in player_dfs.items():
    for season_key, df_season in season_dict.items():
        try:
            all_dfs_v2.append(build_features_v2(pid, season_key, df_season))
        except Exception as e:
            print(f"  ERROR {PLAYERS.get(pid)} {season_key}: {e}")

df_all = pd.concat(all_dfs_v2, ignore_index=True).sort_values('GAME_DATE').reset_index(drop=True)

FEATURES = [
    'season_avg_so_far', 'form_vs_season',
    'roll5_pts', 'roll10_pts',
    'roll5_ast', 'roll5_tov',
    'roll5_min',                    # NEW: workload signal
    'HOME', 'b2b', 'days_rest',
    'opp_def_rating', 'opp_off_rating',   # now rolling, not static season avg
    'games_on_new_team',
]

df_model = df_all.dropna(subset=FEATURES + ['PTS']).copy().reset_index(drop=True)

print(f"v2 dataset: {len(df_model)} rows | {df_model['player_name'].nunique()} players")
print(f"Date range: {df_model['GAME_DATE'].min().date()} → {df_model['GAME_DATE'].max().date()}")
print(f"\nFeatures ({len(FEATURES)}): {FEATURES}")
print(f"\nroll5_min summary:")
print(df_model['roll5_min'].describe().round(1).to_string())


In [ ]:
# Re-run train/test split and retrain with new features
SPLIT_DATE = pd.Timestamp('2023-10-01')
train_mask = df_model['GAME_DATE'] < SPLIT_DATE
test_mask  = ~train_mask

X_train = df_model.loc[train_mask, FEATURES].values
y_train = df_model.loc[train_mask, 'PTS'].values
X_test  = df_model.loc[test_mask,  FEATURES].values
y_test  = df_model.loc[test_mask,  'PTS'].values

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

model = xgb.XGBRegressor(**PARAMS)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

train_rmse_v2 = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
test_rmse_v2  = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae_v2   = mean_absolute_error(y_test, y_pred)
test_r2_v2    = r2_score(y_test, y_pred)

print("\n=== V2 Model Metrics (minutes + rolling ratings) ===")
print(f"  Train RMSE : {train_rmse_v2:.2f} pts")
print(f"  Test  R2   : {test_r2_v2:.3f}")
print(f"  Test  RMSE : {test_rmse_v2:.2f} pts")
print(f"  Test  MAE  : {test_mae_v2:.2f} pts")
print(f"  Gap        : {abs(train_rmse_v2 - test_rmse_v2):.2f} pts",
      '(overfitting)' if train_rmse_v2 < test_rmse_v2 - 1 else '(stable)')

# Baseline comparison
BASELINE_TEST_RMSE = 8.28
BASELINE_TEST_R2   = 0.243
print(f"\n=== Comparison vs Baseline ===")
print(f"  RMSE  : {BASELINE_TEST_RMSE:.2f} → {test_rmse_v2:.2f}  "
      f"({'↓ improved' if test_rmse_v2 < BASELINE_TEST_RMSE else '↑ worse'} "
      f"by {abs(test_rmse_v2 - BASELINE_TEST_RMSE):.2f} pts)")
print(f"  R²    : {BASELINE_TEST_R2:.3f} → {test_r2_v2:.3f}  "
      f"({'↑ improved' if test_r2_v2 > BASELINE_TEST_R2 else '↓ worse'})")

# Per-player RMSE
test_df = df_model[test_mask].copy()
test_df['pred']     = y_pred
test_df['residual'] = y_test - y_pred

per_player = (
    test_df.groupby('player_name')
    .apply(lambda g: pd.Series({
        'games': len(g),
        'avg_pts': g['PTS'].mean().round(1),
        'rmse': round(np.sqrt(mean_squared_error(g['PTS'], g['pred'])), 2),
    }))
    .sort_values('rmse', ascending=False)
)
print(f"\n=== Per-Player RMSE (v2) ===")
print(per_player.to_string())


In [ ]:
# SHAP feature importance for v2 model
explainer_v2 = shap.TreeExplainer(model)
idx_v2       = np.random.choice(len(X_train), min(3000, len(X_train)), replace=False)
shap_vals_v2 = explainer_v2.shap_values(X_train[idx_v2])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance bar
imp_v2 = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp_v2.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance (gain) — v2')
axes[0].set_xlabel('Importance score')

# Actual vs Predicted
axes[1].scatter(y_test, y_pred, alpha=0.15, edgecolors='none', color='steelblue', s=10)
lims = [0, max(y_test.max(), y_pred.max()) + 2]
axes[1].plot(lims, lims, 'r--', lw=1.5)
axes[1].set_xlabel('Actual PTS')
axes[1].set_ylabel('Predicted PTS')
axes[1].set_title(f'Actual vs Predicted — v2  (RMSE={test_rmse_v2:.2f} pts)')

plt.tight_layout()
plt.show()

print("\nSHAP summary plot:")
shap.summary_plot(shap_vals_v2, X_train[idx_v2], feature_names=FEATURES,
                  plot_type='dot', show=True)


## Step 9 — Opponent History Feature + Model Refinement

### What we are adding

**`hist_pts_vs_opp`** — for each game, the player's historical average points scored  
against *this specific opponent* across all prior meetings in our dataset.

**Why this matters:**  
Some players have clear opponent-specific tendencies that pure form averages miss.  
Luka routinely goes off against certain defences; Embiid feasts on teams without  
a rim protector. This feature gives the model a long-run signal: *"given who you're  
playing tonight, how have you historically performed against them?"*

**How it's computed (no leakage):**  
All player-game logs are sorted chronologically. For each (player, opponent) pair,  
we compute an expanding mean of points scored, shifted by 1 game — so every value  
only uses games played *before* the current one. A player's first game against an  
opponent has no prior history; in that case we fall back to `season_avg_so_far`  
as the prior estimate.

---

### What we are removing

**`b2b`** (back-to-back binary flag) is dropped.  
`days_rest` already encodes this information continuously — `days_rest = 1` *is*  
a back-to-back. Including both teaches the model the same signal twice, adding noise  
without information. The SHAP plot confirmed `b2b` had very small marginal SHAP values,  
while `days_rest` already carried the meaningful effect.

---

### Updated feature list (14 → 14, swap b2b for hist_pts_vs_opp)

| Removed | Added |
|---|---|
| `b2b` (redundant with `days_rest`) | `hist_pts_vs_opp` (opponent-specific history) |

In [ ]:
# ── Compute opponent-history feature at the full-dataset level ────────────────
#
# We cannot do this inside build_features_v2 because it processes one player-season
# at a time — we need the player's FULL cross-season history against each opponent.
#
# Approach:
#   1. Sort by (player_id, GAME_DATE) so history is always chronological
#   2. Extract opponent abbreviation from the MATCHUP string
#   3. Group by (player_id, opp_abbr) and compute shift(1).expanding().mean()
#      — every value strictly uses only games BEFORE the current one
#   4. Fill NaN (first ever meeting with an opponent) with season_avg_so_far
#      as the most reasonable prior

df_all = df_all.sort_values(['player_id', 'GAME_DATE']).reset_index(drop=True)
df_all['opp_abbr'] = df_all['MATCHUP'].apply(extract_opp_abbr)

df_all['hist_pts_vs_opp'] = (
    df_all.groupby(['player_id', 'opp_abbr'])['PTS']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Fill first-time matchups with the player's season average so far
df_all['hist_pts_vs_opp'] = df_all['hist_pts_vs_opp'].fillna(df_all['season_avg_so_far'])

# Sanity check: no values should use current-game data (shift=1 guarantees this)
print(f"hist_pts_vs_opp populated: {df_all['hist_pts_vs_opp'].notna().mean()*100:.1f}% of rows")
print(f"\nSample — Luka vs select opponents:")
luka_id = 1629029
luka_opp = (
    df_all[df_all['player_id'] == luka_id]
    [['GAME_DATE', 'opp_abbr', 'PTS', 'hist_pts_vs_opp']]
    .dropna()
    .groupby('opp_abbr')[['PTS', 'hist_pts_vs_opp']].mean().round(1)
    .sort_values('PTS', ascending=False)
    .head(10)
)
print(luka_opp.to_string())


In [ ]:
# ── Rebuild model-ready dataset with updated feature set ──────────────────────
#
# Changes from v2:
#   - Added  : hist_pts_vs_opp (opponent-specific career average, lag-1)
#   - Removed: b2b (redundant — days_rest=1 already captures back-to-backs)
#
# Everything else (PARAMS, train/test split date) stays the same.

FEATURES = [
    'season_avg_so_far',   # who this player is (season identity)
    'form_vs_season',      # hot/cold relative to their own baseline
    'roll5_pts',           # last 5 games form
    'roll10_pts',          # last 10 games form (medium-term trend)
    'roll5_ast',           # assists signal — high assists correlates with high usage
    'roll5_tov',           # turnover rate — proxy for defensive pressure on the player
    'roll5_min',           # workload — catches load management and minute restrictions
    'hist_pts_vs_opp',     # career average vs tonight's specific opponent (lag-1)
    'HOME',                # home court advantage
    'days_rest',           # continuous rest days — covers B2B implicitly (=1) and extra rest
    'opp_def_rating',      # opponent defensive quality (monthly rolling snapshot)
    'opp_off_rating',      # opponent pace/offensive quality → more possessions = more scoring
    'games_on_new_team',   # trade adjustment: performance dips in first N games on new team
]

df_model = df_all.dropna(subset=FEATURES + ['PTS']).copy().reset_index(drop=True)

print(f"Model-ready rows: {len(df_model):,} | Players: {df_model['player_name'].nunique()}")
print(f"\nFeatures ({len(FEATURES)}):")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {f}")


In [ ]:
# ── Retrain with v3 feature set ───────────────────────────────────────────────
#
# Same PARAMS as before — hyperparameters are unchanged so any RMSE movement
# is purely from the feature swap, not tuning.

SPLIT_DATE = pd.Timestamp('2023-10-01')
train_mask = df_model['GAME_DATE'] < SPLIT_DATE
test_mask  = ~train_mask

X_train = df_model.loc[train_mask, FEATURES].values
y_train = df_model.loc[train_mask, 'PTS'].values
X_test  = df_model.loc[test_mask,  FEATURES].values
y_test  = df_model.loc[test_mask,  'PTS'].values

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

model = xgb.XGBRegressor(**PARAMS)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

train_rmse_v3 = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
test_rmse_v3  = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae_v3   = mean_absolute_error(y_test, y_pred)
test_r2_v3    = r2_score(y_test, y_pred)

# ── Progression table ─────────────────────────────────────────────────────────
print("\n=== Model Progression ===")
print(f"  {'Version':<30} {'RMSE':>6}  {'R²':>6}  {'Gap':>6}")
print(f"  {'-'*52}")
print(f"  {'v1 Baseline':30} {'8.28':>6}  {'0.243':>6}  {'0.97':>6}")
print(f"  {'v2 + minutes + rolling ratings':30} {'8.17':>6}  {'0.263':>6}  {'0.88':>6}")
print(f"  {'v3 + opp history, drop b2b':30} {test_rmse_v3:>6.2f}  {test_r2_v3:>6.3f}  {abs(train_rmse_v3-test_rmse_v3):>6.2f}")

# ── Per-player RMSE ───────────────────────────────────────────────────────────
test_df = df_model[test_mask].copy()
test_df['pred']     = y_pred
test_df['residual'] = y_test - y_pred

per_player = (
    test_df.groupby('player_name')
    .apply(lambda g: pd.Series({
        'games': len(g),
        'avg_pts': g['PTS'].mean().round(1),
        'rmse': round(np.sqrt(mean_squared_error(g['PTS'], g['pred'])), 2),
    }))
    .sort_values('rmse', ascending=False)
)
print(f"\n=== Per-Player RMSE (v3) ===")
print(per_player.to_string())


In [ ]:
# ── SHAP analysis — what is each feature actually doing? ─────────────────────
#
# SHAP (SHapley Additive exPlanations) decomposes each prediction into the
# contribution from each feature.
#
# Reading the beeswarm plot:
#   - Each dot = one game
#   - X position = how much that feature pushed the prediction up (right) or down (left)
#   - Colour = feature value (red = high, blue = low)
#
# Example interpretation: if roll5_pts is red and far right, it means
# high recent scoring form pushed the prediction significantly upward.

explainer_v3 = shap.TreeExplainer(model)
idx_v3       = np.random.choice(len(X_train), min(3000, len(X_train)), replace=False)
shap_vals_v3 = explainer_v3.shap_values(X_train[idx_v3])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Feature importance (gain) bar chart
imp_v3 = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
colors = ['#e07b54' if f == 'hist_pts_vs_opp' else 'steelblue' for f in imp_v3.index]
imp_v3.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Feature Importance (gain) — v3\n(orange = newly added feature)', fontsize=11)
axes[0].set_xlabel('Importance score')

# Actual vs Predicted
axes[1].scatter(y_test, y_pred, alpha=0.15, edgecolors='none', color='steelblue', s=10)
lims = [0, max(y_test.max(), y_pred.max()) + 2]
axes[1].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual PTS'); axes[1].set_ylabel('Predicted PTS')
axes[1].set_title(
    f'Actual vs Predicted — v3  (RMSE={test_rmse_v3:.2f} pts, R²={test_r2_v3:.3f})', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

print("SHAP beeswarm (showing directional effect of each feature on each prediction):")
shap.summary_plot(shap_vals_v3, X_train[idx_v3], feature_names=FEATURES,
                  plot_type='dot', show=True)


## Step 10 — Live Next-Game Predictor (2025-26 Season)

### How it works

This predictor runs the trained V2 XGBoost model on a player's **live 2025-26 season data**  
to forecast their points in the next game. The pipeline has four stages:

1. **Retrain v2 model** — `model_pred` is trained explicitly on v2 features so it is  
   decoupled from the v3 experiment cells above. Same `PARAMS`, same train/test split.

2. **Residual distribution** — the model's errors on the test set are stored per player.  
   These residuals describe how uncertain the model is: a player whose test residuals  
   have std ±9 pts is genuinely harder to predict than one with ±6 pts.

3. **Fetch live data** — 2025-26 game logs and team ratings are pulled from `nba_api`.  
   Rolling features are built from the player's actual recent games. The most recent  
   game's feature vector is used as the base for the prediction.

4. **Next-game context override** — forward-looking features (opponent, home/away, rest)  
   are overridden by whatever you specify in the prediction call. The rolling averages  
   (`roll5_pts`, `roll5_min`, etc.) always come from the player's actual recent games.

### Probability of scoring X points

Given a predicted mean `μ` and the empirical residual distribution `e`,  
we estimate **P(score ≥ X) = proportion of residuals where μ + e ≥ X**.  
This is a nonparametric estimate — no normal distribution assumed.  
Per-player residuals are used where 20+ test games are available; otherwise  
the global residual distribution is the fallback.

In [ ]:
# ── Retrain V2 model explicitly as model_pred ─────────────────────────────────
#
# We define FEATURES_PRED here rather than relying on the FEATURES variable,
# which was overwritten by the v3 experiment.  This locks the predictor to v2.

FEATURES_PRED = [
    'season_avg_so_far',   # season identity — who this player is right now
    'form_vs_season',      # recent form relative to their own baseline
    'roll5_pts',           # last 5 games scoring form
    'roll10_pts',          # last 10 games scoring form (medium-term trend)
    'roll5_ast',           # assists — proxy for usage and involvement
    'roll5_tov',           # turnovers — proxy for defensive attention on the player
    'roll5_min',           # recent minutes — catches load management
    'HOME',                # home court advantage (1 = home, 0 = away)
    'b2b',                 # back-to-back flag (1 = played yesterday)
    'days_rest',           # continuous rest days — captures full rest spectrum
    'opp_def_rating',      # opponent defensive quality (monthly rolling snapshot)
    'opp_off_rating',      # opponent pace/offensive quality
    'games_on_new_team',   # games since last trade — captures adjustment period
]

# df_all was built by build_features_v2 and contains all these columns
df_pred_base = df_all.dropna(subset=FEATURES_PRED + ['PTS']).copy().reset_index(drop=True)

SPLIT_DATE  = pd.Timestamp('2023-10-01')
tr_mask_p   = df_pred_base['GAME_DATE'] < SPLIT_DATE
te_mask_p   = ~tr_mask_p

X_tr_p = df_pred_base.loc[tr_mask_p, FEATURES_PRED].values
y_tr_p = df_pred_base.loc[tr_mask_p, 'PTS'].values
X_te_p = df_pred_base.loc[te_mask_p, FEATURES_PRED].values
y_te_p = df_pred_base.loc[te_mask_p, 'PTS'].values

model_pred = xgb.XGBRegressor(**PARAMS)
model_pred.fit(X_tr_p, y_tr_p)
y_hat_te   = model_pred.predict(X_te_p)

global_residuals = y_te_p - y_hat_te   # shape (n_test,)

test_rmse_pred = np.sqrt(mean_squared_error(y_te_p, y_hat_te))
print(f"model_pred (v2) — Test RMSE: {test_rmse_pred:.2f} pts")
print(f"Global residual  mean: {global_residuals.mean():.2f} pts  "
      f"std: {global_residuals.std():.2f} pts")

# ── Per-player residual distributions ─────────────────────────────────────────
# Using per-player residuals gives more calibrated probabilities for high-variance
# scorers (Curry, Booker) vs. consistent players (Gobert, CP3).
# Minimum 20 test games required for a player-specific distribution.

df_te_p = df_pred_base[te_mask_p].copy().reset_index(drop=True)
df_te_p['resid'] = global_residuals

player_residuals = {}
for pid, grp in df_te_p.groupby('player_id'):
    if len(grp) >= 20:
        player_residuals[pid] = grp['resid'].values

print(f"\nPer-player residual distributions available for "
      f"{len(player_residuals)}/{df_te_p['player_id'].nunique()} players")
print("\nPer-player residual std (prediction uncertainty):")
resid_stds = (
    pd.Series({PLAYERS.get(pid, pid): r.std() for pid, r in player_residuals.items()})
    .sort_values(ascending=False)
    .round(2)
)
print(resid_stds.to_string())


In [ ]:
# ── Fetch 2025-26 season team ratings ────────────────────────────────────────
#
# We need current-season opponent ratings for the prediction context.
# These are fetched once, added to rolling_ratings and team_ratings,
# and cached in memory for the session (no file write needed — just re-run
# this cell at the start of each session to get fresh ratings).

from nba_api.stats.endpoints import leaguedashteamstats as _ldt

CURRENT_SEASON     = '2025-26'
CURRENT_SEASON_KEY = '2025_26'

df_curr_ratings = _ldt.LeagueDashTeamStats(
    season=CURRENT_SEASON,
    measure_type_detailed_defense='Advanced',
).get_data_frames()[0]

_snapshot_key = pd.Timestamp.today().strftime('%Y-%m-%d')

# Add to rolling_ratings so build_features_v2 can look up opponent ratings
if CURRENT_SEASON_KEY not in rolling_ratings:
    rolling_ratings[CURRENT_SEASON_KEY] = {}

rolling_ratings[CURRENT_SEASON_KEY][_snapshot_key] = {
    int(row['TEAM_ID']): {'DEF': row['DEF_RATING'], 'OFF': row['OFF_RATING']}
    for _, row in df_curr_ratings.iterrows()
}

# Also add to team_ratings as the season-level fallback
team_ratings[CURRENT_SEASON_KEY] = {
    int(row['TEAM_ID']): {'DEF': row['DEF_RATING'], 'OFF': row['OFF_RATING']}
    for _, row in df_curr_ratings.iterrows()
}

# Build abbr -> ratings lookup for the prediction function
_id_to_abbr_map = {t['id']: t['abbreviation'] for t in nba_teams.get_teams()}
CURRENT_TEAM_RATINGS = {}   # {abbr: {DEF, OFF}}  — used in predict_next_game
for tid, ratings in team_ratings[CURRENT_SEASON_KEY].items():
    abbr = _id_to_abbr_map.get(tid)
    if abbr:
        CURRENT_TEAM_RATINGS[abbr] = ratings

print(f"2025-26 team ratings loaded ({len(CURRENT_TEAM_RATINGS)} teams, snapshot: {_snapshot_key})")
print(f"\nTop-5 toughest defences (highest DEF rating = easier to score against):")
def_rank = sorted(CURRENT_TEAM_RATINGS.items(), key=lambda x: x[1]['DEF'])
print("  Hardest to score against:")
for abbr, r in def_rank[:5]:
    print(f"    {abbr:<5}  DEF={r['DEF']:.1f}")
print("  Easiest to score against:")
for abbr, r in def_rank[-5:]:
    print(f"    {abbr:<5}  DEF={r['DEF']:.1f}")

time.sleep(0.3)

# ── Function: fetch a player's 2025-26 game logs and build v2 features ────────
def fetch_player_current_season(player_id):
    """
    Fetches 2025-26 game logs for player_id and returns a feature-engineered
    DataFrame using build_features_v2.  Raises ValueError if no data found.
    """
    logs = playergamelogs.PlayerGameLogs(
        player_id_nullable=player_id,
        season_nullable=CURRENT_SEASON,
    ).get_data_frames()[0]

    if logs.empty:
        raise ValueError(f"No {CURRENT_SEASON} data found for player ID {player_id}")

    keep = [c for c in ['GAME_DATE', 'TEAM_ABBREVIATION', 'MATCHUP', 'WL',
                         'MIN', 'PTS', 'REB', 'AST', 'TOV', 'PLUS_MINUS']
            if c in logs.columns]
    df = logs[keep].copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    return build_features_v2(player_id, CURRENT_SEASON_KEY, df)

print("\nfetch_player_current_season() defined.")


In [ ]:
# ── predict_next_game — main prediction function ─────────────────────────────
#
# Parameters
# ----------
# player_id      : int   NBA player ID  (see PLAYERS dict for valid IDs)
# opponent_abbr  : str   Opponent team abbreviation e.g. 'BOS', 'LAL', 'GSW'
#                        Pass None to use league-average opponent ratings
# is_home        : bool  True = home game, False = away game
# days_rest_next : int   Days since last game (1 = back-to-back, 2 = normal, 3+ = rested)
# target_pts     : float Compute P(score >= target_pts)  — set to None to skip
# plot           : bool  Show probability density plot of predicted scoring range
#
# How the probability is computed
# --------------------------------
# 1. Get the point prediction μ from model_pred
# 2. Shift the empirical residual distribution by μ → distribution of possible outcomes
# 3. P(score ≥ X) = fraction of (μ + residuals) that exceed X
# 4. The 80% interval is the [p10, p90] of the same shifted distribution

from scipy.stats import gaussian_kde

def predict_next_game(player_id, opponent_abbr=None, is_home=True,
                      days_rest_next=2, target_pts=None, plot=True):

    player_name = PLAYERS.get(player_id, f"Player {player_id}")

    # 1. Fetch and build features from live 2025-26 data
    print(f"Fetching {CURRENT_SEASON} logs for {player_name}...")
    df_live = fetch_player_current_season(player_id)
    df_live = df_live.dropna(subset=FEATURES_PRED)

    if len(df_live) < 5:
        print(f"  Only {len(df_live)} usable rows — not enough to make a reliable prediction.")
        return None

    # 2. Use most recent game's rolling features as the base
    #    (these encode the player's CURRENT form — roll5_pts, season_avg, etc.)
    last_row   = df_live.iloc[-1]
    feat_dict  = {f: float(last_row[f]) for f in FEATURES_PRED}

    # 3. Override forward-looking context with next-game inputs
    feat_dict['HOME']      = 1.0 if is_home else 0.0
    feat_dict['days_rest'] = float(days_rest_next)
    feat_dict['b2b']       = 1.0 if days_rest_next == 1 else 0.0

    if opponent_abbr and opponent_abbr.upper() in CURRENT_TEAM_RATINGS:
        opp = CURRENT_TEAM_RATINGS[opponent_abbr.upper()]
        feat_dict['opp_def_rating'] = opp['DEF']
        feat_dict['opp_off_rating'] = opp['OFF']
        opp_label = opponent_abbr.upper()
    else:
        # League-average ratings
        feat_dict['opp_def_rating'] = float(np.mean([v['DEF'] for v in CURRENT_TEAM_RATINGS.values()]))
        feat_dict['opp_off_rating'] = float(np.mean([v['OFF'] for v in CURRENT_TEAM_RATINGS.values()]))
        opp_label = 'league average'

    # 4. Point prediction
    X_new      = np.array([[feat_dict[f] for f in FEATURES_PRED]])
    point_pred = float(max(0, model_pred.predict(X_new)[0]))

    # 5. Outcome distribution = point_pred + residuals
    resids    = player_residuals.get(player_id, global_residuals)
    outcomes  = point_pred + resids
    outcomes  = outcomes[outcomes >= 0]

    lo80 = float(np.percentile(outcomes, 10))
    hi80 = float(np.percentile(outcomes, 90))

    # 6. Probability that actual score exceeds target
    prob_val  = None
    prob_line = ""
    if target_pts is not None:
        prob_val  = float((outcomes >= target_pts).mean())
        prob_line = f"  P(score ≥ {int(target_pts)} pts)   : {prob_val*100:.1f}%"

    # 7. Print summary
    games_played = len(df_live)
    recent = df_live.tail(5)[['GAME_DATE', 'MATCHUP', 'PTS']].copy()
    recent['GAME_DATE'] = recent['GAME_DATE'].dt.strftime('%Y-%m-%d')

    print(f"\n{'='*58}")
    print(f"  Next Game Prediction — {player_name}")
    print(f"{'='*58}")
    print(f"  Opponent              : {opp_label}")
    print(f"  Location              : {'Home' if is_home else 'Away'}")
    print(f"  Days rest             : {days_rest_next}")
    print(f"  Games played (season) : {games_played}")
    print(f"{'─'*58}")
    print(f"  Predicted pts         : {point_pred:.1f}")
    print(f"  80% interval          : {lo80:.1f} – {hi80:.1f} pts")
    print(f"{'─'*58}")
    print(f"  Season avg so far     : {last_row['season_avg_so_far']:.1f} pts")
    print(f"  Roll-5 form           : {last_row['roll5_pts']:.1f} pts")
    print(f"  Roll-10 form          : {last_row['roll10_pts']:.1f} pts")
    print(f"  Roll-5 minutes        : {last_row['roll5_min']:.1f} min")
    if prob_line:
        print(f"{'─'*58}")
        print(prob_line)
    print(f"{'─'*58}")
    print(f"  Last 5 games:")
    for _, r in recent.iterrows():
        print(f"    {r['GAME_DATE']}  {str(r['MATCHUP']):<30}  {int(r['PTS'])} pts")
    print(f"{'='*58}")

    # 8. Optional probability density plot
    if plot:
        fig, ax = plt.subplots(figsize=(10, 4))
        kde = gaussian_kde(outcomes, bw_method=0.4)
        x_range = np.linspace(max(0, outcomes.min() - 5), outcomes.max() + 5, 300)
        y_vals  = kde(x_range)

        ax.plot(x_range, y_vals, color='steelblue', lw=2, label='Predicted distribution')
        ax.fill_between(x_range, y_vals,
                        where=(x_range >= lo80) & (x_range <= hi80),
                        alpha=0.25, color='steelblue', label=f'80% interval ({lo80:.0f}–{hi80:.0f} pts)')

        ax.axvline(point_pred, color='navy', linestyle='--', lw=2,
                   label=f'Prediction: {point_pred:.1f} pts')

        if target_pts is not None:
            ax.axvline(target_pts, color='crimson', linestyle=':', lw=2,
                       label=f'Target ≥{int(target_pts)} pts  ({prob_val*100:.1f}%)')
            ax.fill_between(x_range, y_vals,
                            where=(x_range >= target_pts),
                            alpha=0.2, color='crimson')

        ax.set_xlabel('Points scored', fontsize=12)
        ax.set_ylabel('Probability density', fontsize=12)
        ax.set_title(f'{player_name} — Next Game Prediction vs {opp_label}', fontsize=13)
        ax.set_xlim(left=0)
        ax.legend(fontsize=10)
        plt.tight_layout()
        plt.show()

    return {
        'player':        player_name,
        'predicted_pts': round(point_pred, 1),
        'interval':      (round(lo80, 1), round(hi80, 1)),
        'prob_exceed':   round(prob_val * 100, 1) if prob_val is not None else None,
    }

print("predict_next_game() ready.")
print(f"\nRegistered players:")
for pid, name in sorted(PLAYERS.items(), key=lambda x: x[1]):
    print(f"  {pid:>8}  {name}")


In [ ]:
# ── Run the predictor ─────────────────────────────────────────────────────────
#
# Edit the variables below and re-run this cell.
#
# PLAYER_ID   : pick any ID from the list printed above
# OPPONENT    : 3-letter team abbreviation, or None for league average
#               (run: print(list(CURRENT_TEAM_RATINGS.keys())) to see all valid codes)
# HOME_GAME   : True = home court, False = away
# DAYS_REST   : 1 = back-to-back, 2 = one day off (typical), 3+ = extra rest
# TARGET_PTS  : the scoring threshold for the probability question,
#               e.g. 30 → "what is P(score ≥ 30)?"  Set to None to skip.

PLAYER_ID  = 1629029   # Luka Doncic
OPPONENT   = 'BOS'     # Boston Celtics
HOME_GAME  = True
DAYS_REST  = 2
TARGET_PTS = 30

result = predict_next_game(
    player_id      = PLAYER_ID,
    opponent_abbr  = OPPONENT,
    is_home        = HOME_GAME,
    days_rest_next = DAYS_REST,
    target_pts     = TARGET_PTS,
)
